# Benchmark — Le français dans les datasets RLHF publics

Audit quantitatif de la représentation du français dans 3 datasets RLHF publics majeurs :

- **UltraFeedback** (openbmb/UltraFeedback) — ~64 000 prompts
- **OpenAssistant OASST1** (OpenAssistant/oasst1) — ~31 500 prompts utilisateur
- **Anthropic HH-RLHF** (Anthropic/hh-rlhf) — échantillon de 30 000 sur ~160 000 dialogues

**Méthode :** détection automatique de langue via [lingua-py](https://github.com/pemistahl/lingua-py).

**Reproduire :** 
1. Ouvrir ce notebook dans Google Colab
2. Créer un token Hugging Face (lecture seule) sur huggingface.co/settings/tokens
3. Coller le token dans la cellule de login ci-dessous
4. Run All

**Auteur :** Horeb N'DINGA — STEF


In [ ]:
# Installation des librairies nécessaires
!pip install -q datasets huggingface_hub lingua-language-detector pandas matplotlib tqdm

print("✅ Installation terminée")

In [ ]:
# Connexion à Hugging Face
from huggingface_hub import login

# REMPLACE PAR TON PROPRE TOKEN HUGGING FACE (Read access)
# Crée-en un sur https://huggingface.co/settings/tokens
TON_TOKEN = "hf_REMPLACE_PAR_TON_TOKEN"

login(token=TON_TOKEN)
print("✅ Connecté à Hugging Face")


In [ ]:
# Setup du détecteur de langue avec lingua-py
from lingua import Language, LanguageDetectorBuilder

# On crée un détecteur qui peut reconnaître toutes les langues
detector = LanguageDetectorBuilder.from_all_languages().with_low_accuracy_mode().build()

# Test rapide
test_fr = detector.detect_language_of("Bonjour, comment allez-vous aujourd'hui ?")
test_en = detector.detect_language_of("Hello, how are you doing today?")

print(f"Test français : {test_fr}")
print(f"Test anglais : {test_en}")
print("✅ Détecteur de langue prêt")

In [ ]:
# Chargement du dataset UltraFeedback
from datasets import load_dataset

print("⏳ Téléchargement de UltraFeedback en cours... (peut prendre 2-5 min)")

dataset = load_dataset("openbmb/UltraFeedback", split="train")

print(f"✅ Dataset chargé : {len(dataset)} exemples au total")
print(f"Colonnes disponibles : {dataset.column_names}")
print(f"\nExemple d'un prompt :")
print(dataset[0]["instruction"][:200])

In [ ]:
# Détection de langue sur tous les prompts d'UltraFeedback (avec lingua)
from collections import Counter
from tqdm import tqdm

# On collecte la langue de chaque prompt
langues = []
prompts = dataset["instruction"]

print(f"⏳ Analyse de {len(prompts)} prompts...")

for prompt in tqdm(prompts):
    # On nettoie le texte
    texte_nettoye = str(prompt).replace("\n", " ").strip()

    # On skip les textes vides ou trop courts
    if len(texte_nettoye) < 10:
        langues.append("trop_court")
        continue

    try:
        result = detector.detect_language_of(texte_nettoye)
        if result is None:
            langues.append("indetermine")
        else:
            # On récupère le code ISO 639-1 (fr, en, etc.) en minuscule
            langues.append(result.iso_code_639_1.name.lower())
    except Exception as e:
        langues.append("erreur")

# Comptage
compteur = Counter(langues)
total = len(langues)

print(f"\n=== RÉSULTATS UltraFeedback ===")
print(f"Total exemples analysés : {total}")
print(f"\nTop 10 langues détectées :")
for langue, count in compteur.most_common(10):
    pourcentage = 100 * count / total
    print(f"  {langue} : {count} exemples ({pourcentage:.2f}%)")

# On stocke les résultats pour le récap final
resultats_ultrafeedback = {
    "dataset": "UltraFeedback",
    "total": total,
    "fr": compteur.get("fr", 0),
    "en": compteur.get("en", 0),
    "fr_percent": 100 * compteur.get("fr", 0) / total,
    "en_percent": 100 * compteur.get("en", 0) / total,
}

print(f"\n✅ %FR = {resultats_ultrafeedback['fr_percent']:.2f}%")
print(f"✅ %EN = {resultats_ultrafeedback['en_percent']:.2f}%")

In [ ]:
# Chargement du dataset OpenAssistant OASST1
print("⏳ Téléchargement de OASST1 en cours... (peut prendre 1-3 min)")

dataset_oasst = load_dataset("OpenAssistant/oasst1", split="train")

print(f"✅ Dataset chargé : {len(dataset_oasst)} exemples au total")
print(f"Colonnes disponibles : {dataset_oasst.column_names}")
print(f"\nExemple d'un prompt :")
print(dataset_oasst[0])

In [ ]:
# On filtre uniquement les prompts utilisateur (pas les réponses de l'assistant)
prompts_oasst = dataset_oasst.filter(lambda x: x["role"] == "prompter")

print(f"✅ Nombre de prompts utilisateur : {len(prompts_oasst)}")
print(f"\nExemple :")
print(f"  Texte : {prompts_oasst[0]['text'][:150]}...")
print(f"  Langue déclarée : {prompts_oasst[0]['lang']}")

In [ ]:
# Détection de langue sur les prompts d'OASST1
from collections import Counter
from tqdm import tqdm

# On collecte deux choses : la langue déclarée et la langue détectée
langues_declarees = []
langues_detectees = []

textes = prompts_oasst["text"]
langs_field = prompts_oasst["lang"]

print(f"⏳ Analyse de {len(textes)} prompts...")

for texte, lang_declaree in tqdm(zip(textes, langs_field), total=len(textes)):
    # Langue déclarée par l'auteur
    langues_declarees.append(lang_declaree if lang_declaree else "non_specifie")

    # Langue détectée par lingua
    texte_nettoye = str(texte).replace("\n", " ").strip()

    if len(texte_nettoye) < 10:
        langues_detectees.append("trop_court")
        continue

    try:
        result = detector.detect_language_of(texte_nettoye)
        if result is None:
            langues_detectees.append("indetermine")
        else:
            langues_detectees.append(result.iso_code_639_1.name.lower())
    except Exception:
        langues_detectees.append("erreur")

# Comptages
compteur_declare = Counter(langues_declarees)
compteur_detecte = Counter(langues_detectees)
total = len(textes)

print(f"\n=== RÉSULTATS OASST1 ===")
print(f"Total prompts analysés : {total}")

print(f"\n--- Langue DÉCLARÉE par l'auteur (top 10) ---")
for langue, count in compteur_declare.most_common(10):
    pourcentage = 100 * count / total
    print(f"  {langue} : {count} prompts ({pourcentage:.2f}%)")

print(f"\n--- Langue DÉTECTÉE par lingua (top 10) ---")
for langue, count in compteur_detecte.most_common(10):
    pourcentage = 100 * count / total
    print(f"  {langue} : {count} prompts ({pourcentage:.2f}%)")

# On stocke les résultats
resultats_oasst = {
    "dataset": "OASST1",
    "total": total,
    "fr_declare": compteur_declare.get("fr", 0),
    "en_declare": compteur_declare.get("en", 0),
    "fr_detecte": compteur_detecte.get("fr", 0),
    "en_detecte": compteur_detecte.get("en", 0),
    "fr_percent_declare": 100 * compteur_declare.get("fr", 0) / total,
    "en_percent_declare": 100 * compteur_declare.get("en", 0) / total,
    "fr_percent_detecte": 100 * compteur_detecte.get("fr", 0) / total,
    "en_percent_detecte": 100 * compteur_detecte.get("en", 0) / total,
}

print(f"\n✅ %FR (déclaré) = {resultats_oasst['fr_percent_declare']:.2f}%")
print(f"✅ %FR (détecté) = {resultats_oasst['fr_percent_detecte']:.2f}%")
print(f"✅ %EN (déclaré) = {resultats_oasst['en_percent_declare']:.2f}%")
print(f"✅ %EN (détecté) = {resultats_oasst['en_percent_detecte']:.2f}%")

In [ ]:
from datasets import load_dataset

# Chargement du dataset Anthropic HH-RLHF
print("⏳ Téléchargement de HH-RLHF en cours... (peut prendre 2-5 min)")

dataset_hh = load_dataset("Anthropic/hh-rlhf", split="train")

print(f"✅ Dataset chargé : {len(dataset_hh)} exemples au total")
print(f"Colonnes disponibles : {dataset_hh.column_names}")
print(f"\nExemple :")
print(f"  Chosen (début) : {dataset_hh[0]['chosen'][:200]}...")


In [ ]:
# Extraction des prompts humains de HH-RLHF + détection de langue
import re
import random
from collections import Counter
from tqdm import tqdm

# Pour gagner du temps, on échantillonne 30 000 exemples
random.seed(42)

if len(dataset_hh) > 30000:
    indices_echantillon = random.sample(range(len(dataset_hh)), 30000)
    echantillon = dataset_hh.select(indices_echantillon)
    print(f"⏳ Échantillon aléatoire de 30 000 sur {len(dataset_hh)} exemples")
else:
    echantillon = dataset_hh
    print(f"⏳ Analyse de tous les {len(dataset_hh)} exemples")

langues = []

for exemple in tqdm(echantillon):
    dialogue = exemple["chosen"]

    match = re.search(r"Human:\s*(.*?)\s*Assistant:", dialogue, re.DOTALL)

    if match:
        prompt_humain = match.group(1).strip()
    else:
        prompt_humain = dialogue[:500]

    texte_nettoye = prompt_humain.replace("\n", " ").strip()

    if len(texte_nettoye) < 10:
        langues.append("trop_court")
        continue

    try:
        result = detector.detect_language_of(texte_nettoye)
        if result is None:
            langues.append("indetermine")
        else:
            langues.append(result.iso_code_639_1.name.lower())
    except Exception:
        langues.append("erreur")

compteur = Counter(langues)
total = len(langues)

print(f"\n=== RÉSULTATS HH-RLHF ===")
print(f"Total exemples analysés : {total}")
print(f"\nTop 10 langues détectées :")
for langue, count in compteur.most_common(10):
    pourcentage = 100 * count / total
    print(f"  {langue} : {count} exemples ({pourcentage:.2f}%)")

resultats_hh = {
    "dataset": "Anthropic HH-RLHF",
    "total": total,
    "fr": compteur.get("fr", 0),
    "en": compteur.get("en", 0),
    "fr_percent": 100 * compteur.get("fr", 0) / total,
    "en_percent": 100 * compteur.get("en", 0) / total,
}

print(f"\n✅ %FR = {resultats_hh['fr_percent']:.2f}%")
print(f"✅ %EN = {resultats_hh['en_percent']:.2f}%")

In [ ]:
# Construction du tableau récap final
import pandas as pd

# On normalise les 3 dictionnaires en lignes comparables
recap = pd.DataFrame([
    {
        "Dataset": "UltraFeedback",
        "Total_Prompts": resultats_ultrafeedback["total"],
        "Nb_FR": resultats_ultrafeedback["fr"],
        "Nb_EN": resultats_ultrafeedback["en"],
        "Pourcent_FR": resultats_ultrafeedback["fr_percent"],
        "Pourcent_EN": resultats_ultrafeedback["en_percent"],
    },
    {
        "Dataset": "OpenAssistant OASST1",
        "Total_Prompts": resultats_oasst["total"],
        "Nb_FR": resultats_oasst["fr_detecte"],
        "Nb_EN": resultats_oasst["en_detecte"],
        "Pourcent_FR": resultats_oasst["fr_percent_detecte"],
        "Pourcent_EN": resultats_oasst["en_percent_detecte"],
    },
    {
        "Dataset": "Anthropic HH-RLHF",
        "Total_Prompts": resultats_hh["total"],
        "Nb_FR": resultats_hh["fr"],
        "Nb_EN": resultats_hh["en"],
        "Pourcent_FR": resultats_hh["fr_percent"],
        "Pourcent_EN": resultats_hh["en_percent"],
    },
])

# Affichage propre
print("=== TABLEAU RÉCAP FINAL ===\n")
print(recap.to_string(index=False))

# Export CSV
recap.to_csv("benchmark_french_rlhf_results.csv", index=False)
print("\n✅ CSV exporté : benchmark_french_rlhf_results.csv")

# Stats globales
total_fr = recap["Nb_FR"].sum()
total_en = recap["Nb_EN"].sum()
total_all = recap["Total_Prompts"].sum()

print(f"\n=== STATS GLOBALES (3 datasets) ===")
print(f"Total prompts analysés : {total_all}")
print(f"Total prompts FR : {total_fr}")
print(f"Total prompts EN : {total_en}")
print(f"Ratio EN/FR : {total_en / max(total_fr, 1):.1f} pour 1")
print(f"%FR global : {100 * total_fr / total_all:.2f}%")
print(f"%EN global : {100 * total_en / total_all:.2f}%")

In [ ]:
# Génération des graphiques matplotlib
import matplotlib.pyplot as plt
import numpy as np

# On utilise le tableau recap déjà créé
datasets_names = recap["Dataset"].tolist()
pourcent_fr = recap["Pourcent_FR"].tolist()
pourcent_en = recap["Pourcent_EN"].tolist()
nb_fr = recap["Nb_FR"].tolist()
nb_en = recap["Nb_EN"].tolist()

# ===========================================
# Graphique 1 : Pourcentages FR vs EN par dataset
# ===========================================
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(datasets_names))
width = 0.35

bars1 = ax.bar(x - width/2, pourcent_en, width, label='Anglais', color='#3498db')
bars2 = ax.bar(x + width/2, pourcent_fr, width, label='Français', color='#e74c3c')

ax.set_ylabel('Pourcentage des prompts (%)', fontsize=12)
ax.set_title('Part du français vs anglais dans 3 datasets RLHF publics', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(datasets_names, fontsize=10)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Annotations sur les barres
for bar, val in zip(bars1, pourcent_en):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
for bar, val in zip(bars2, pourcent_fr):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.2f}%', ha='center', fontsize=10, fontweight='bold', color='#c0392b')

plt.tight_layout()
plt.savefig('graph1_pourcentages.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Graphique 1 sauvegardé : graph1_pourcentages.png")

# ===========================================
# Graphique 2 : Volume absolu de prompts FR
# ===========================================
fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(datasets_names, nb_fr, color='#e74c3c', edgecolor='#c0392b', linewidth=2)

ax.set_ylabel('Nombre absolu de prompts en français', fontsize=12)
ax.set_title('Volume absolu de prompts français dans 3 datasets RLHF publics', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Annotations avec valeur + total dataset
for bar, val_fr, total in zip(bars, nb_fr, recap["Total_Prompts"].tolist()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{val_fr}\n(sur {total:,})', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('graph2_volume_absolu_fr.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Graphique 2 sauvegardé : graph2_volume_absolu_fr.png")

# ===========================================
# Graphique 3 : Ratio EN/FR par dataset
# ===========================================
fig, ax = plt.subplots(figsize=(10, 6))

ratios = [en / max(fr, 1) for en, fr in zip(nb_en, nb_fr)]

bars = ax.bar(datasets_names, ratios, color='#34495e', edgecolor='#2c3e50', linewidth=2)

ax.set_ylabel('Ratio Anglais / Français', fontsize=12)
ax.set_title('Combien de prompts anglais pour 1 prompt français ?', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, ratios):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{val:.0f}× plus', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('graph3_ratio_en_fr.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Graphique 3 sauvegardé : graph3_ratio_en_fr.png")

print("\n=== TOUS LES GRAPHIQUES GÉNÉRÉS ===")